DMP PDF
  ↓
1. pdfplumber extraction
  ↓
2. rule-based structure detection
  ↓
3. build narrative JSON
  ↓
4. save final JSON


# Part 1 — Imports

In [81]:
from pathlib import Path
import pandas as pd

from dmpbridge.pdf.pdfplumber_extractor import save_pdfplumber_outputs
from dmpbridge.processing.structure_detector import detect_structure
from dmpbridge.processing.structure_json_builder import save_narrative_json

# Part 2 — Paths

In [82]:
project_root = Path.cwd().parent

pdf_path = project_root / "data" / "raw_pdfs" / "sample11.pdf"
skeleton_path = project_root / "schemas" / "rda_dmp_dmptool_extension_skeleton.json"

pdfplumber_json_path = project_root / "data" / "pdfplumber_blocks" / f"{pdf_path.stem}.json"
csv_output_path = project_root / "outputs" / "debug" / f"{pdf_path.stem}_structured_lines.csv"
final_json_path = project_root / "data" / "structure_json" / f"{pdf_path.stem}_pdfplumber.json"

print("PDF exists:", pdf_path.exists())
print("Skeleton exists:", skeleton_path.exists())

PDF exists: True
Skeleton exists: True


# Part 3 — Run pdfplumber extraction

In [83]:
blocks = save_pdfplumber_outputs(pdf_path)

print("Extracted lines:", len(blocks))
print("Saved pdfplumber JSON:", pdfplumber_json_path.exists())

[2026-05-15 11:34:28] Extracting line-level text with pdfplumber: sample11.pdf
[2026-05-15 11:34:28] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_blocks\sample11.json
[2026-05-15 11:34:28] Saved extracted text: C:\Users\Nahid\dmpbridge\data\extracted_text\sample11.txt
Extracted lines: 94
Saved pdfplumber JSON: True


# Part 4 — Run rule-based structure detection

In [84]:

structured_blocks = detect_structure(blocks)

df = pd.DataFrame(structured_blocks)
print("Detected format:", df["document_format"].iloc[0])
print(df["label"].value_counts())

df[[
    "page",
    "line_order",
    "text",
    "avg_font_size",
    "is_bold",
    "label",
    "document_format"
]].head(100)

Detected format: unknown
label
content    94
Name: count, dtype: int64


,page,line_order,text,avg_font_size,is_bold,label,document_format
0,1,1,(cid:15)(cid:5)(cid:19)(cid:17)(cid:22)(cid:12...,10.56,True,content,unknown
1,1,2,(cid:24)(cid:48)(cid:33)(cid:44)(cid:48)(cid:3...,10.56,True,content,unknown
2,1,3,(cid:27)(cid:31)(cid:21)(cid:1)(cid:19)(cid:44...,10.56,False,content,unknown
3,1,4,(cid:27)(cid:31)(cid:21)(cid:1)(cid:29)(cid:50...,10.56,False,content,unknown
4,1,5,(cid:16)(cid:29)(cid:46)(cid:29)(cid:1)(cid:27...,10.56,True,content,unknown
...,...,...,...,...,...,...,...
89,2,42,(cid:19)(cid:53)(cid:44)(cid:57)(cid:40)(cid:1...,10.56,False,content,unknown
90,2,43,(cid:41)(cid:50)(cid:53)(cid:1)(cid:36)(cid:38...,10.56,False,content,unknown
91,2,44,(cid:51)(cid:40)(cid:53)(cid:48)(cid:44)(cid:5...,10.56,False,content,unknown
92,2,45,(cid:1),10.56,False,content,unknown


# Part 5 — Inspect extracted lines

In [85]:
print("Detected document format:", df["document_format"].iloc[0])
print(df["label"].value_counts())

Detected document format: unknown
label
content    94
Name: count, dtype: int64


# Part 6 — Save CSV debug file

In [86]:
csv_output_path.parent.mkdir(parents=True, exist_ok=True)

df[[
    "page",
    "line_order",
    "text",
    "avg_font_size",
    "is_bold",
    "label",
    "document_format"
]].to_csv(csv_output_path, index=False, encoding="utf-8")

print("Saved CSV:", csv_output_path)

Saved CSV: c:\Users\Nahid\dmpbridge\outputs\debug\sample11_structured_lines.csv


# Part 7 — Build narrative JSON

In [87]:
final_json = save_narrative_json(
    structured_blocks=structured_blocks,
    output_path=final_json_path,
    skeleton_path=skeleton_path
)

sections = final_json["narrative"]["template"]["section"]

print("Saved JSON:", final_json_path)
print("Number of sections:", len(sections))

for sec in sections:
    print(sec["order"], sec["title"], "| questions:", len(sec["question"]))

[2026-05-15 11:34:28] Saved narrative JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample11_pdfplumber.json
Saved JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample11_pdfplumber.json
Number of sections: 0


# Part 8 — Inspect one section

In [88]:
sections[0]

IndexError: list index out of range